# Project Presentation

This notebook focuses on forecasting the UV Index using historical UV data combined with meteorological and atmospheric variables (e.g., cloud cover, ozone, aerosols, temperature). In addition to prediction, the analysis explores correlations between UV exposure and health-related risks to support risk categorization and preventive insights.

# Loading

## Libraries

### General

In [ ]:
# Standard library
import ast
import csv
import os
import pickle
import re
import sys
import traceback
import warnings
from datetime import datetime
from pathlib import Path
from urllib.parse import parse_qs, urlencode, urlparse, urlunparse

# Third-party: scientific & data
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from scipy import stats
import seaborn as sns

### Models

In [ ]:
from sklearn.ensemble import (
AdaBoostRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

# Third-party: gradient boosting libraries
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

# Third-party: deep learning (TensorFlow / Keras)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import Bidirectional, Dense, Dropout, GRU, LSTM
from tensorflow.keras.models import Sequential

In [ ]:
from utils import preprocess
from utils import plot
from utils import ts
from utils import ml
from utils import dl
from utils import boosters

## Data

In [ ]:
algeria_df = pd.read_csv('data/algeria/Algeria.csv')

# EDA

### General View

In [ ]:
print("Data Info:")
algeria_df.info()

In [ ]:
print("\nDescriptive Statistics:")
print(algeria_df.describe())

In [ ]:
print("\nCorrelation Matrix:")
corr = algeria_df.select_dtypes(include=[np.number]).corr()

plt.figure(figsize=(14, 10))
sns.heatmap(
    corr,
    cmap="coolwarm",
    center=0,
    annot=False,
    linewidths=0.5
)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
print("Null counts per column:")
print(algeria_df.isnull().sum())

## Preprocessing

### Cleaning

In [ ]:
# Columns to drop (irrelevant + redundant for UV–health modeling)
cols_to_drop = [
    # Wind direction (irrelevant)
    "WD2M", "WD10M", "WD50M",
    
    # High-altitude / weak relevance
    "WS50M",
    
    # Radiation redundancy / low UV relevance
    "ALLSKY_SFC_LW_DWN",
    "ALLSKY_KT",
    "CLRSKY_SFC_SW_DWN",
    
    # Redundant temperature features
    "T2MWET",
    "T2M_RANGE",
    
    # Optional weak feature
    "WS10M",
]

algeria_df_clean = algeria_df.drop(columns=cols_to_drop, errors="ignore")
print("\nColumns after dropping irrelevant/redundant features:")
print(algeria_df_clean.columns)

### Converting Columns

In [ ]:
algeria_df_clean["Date"] = pd.to_datetime(algeria_df_clean["Date"])

### Nulls

In [ ]:
print("Null counts per column:")
print(algeria_df_clean.isnull().sum())

In [ ]:
algeria_df_full = algeria_df_clean.copy()
algeria_df = algeria_df_full[
    algeria_df_full["ALLSKY_SFC_UV_INDEX"].notna()
]
print(f"Data shape after removing rows with missing target values: {algeria_df.shape}")
print(algeria_df.isnull().sum())

In [ ]:
print("\nDescriptive Statistics:")
print(algeria_df.describe())

## Plotting

In [ ]:
# Plot all remaining numeric columns over time in a grid of subplots
numeric_cols = algeria_df.select_dtypes(include=[np.number]).columns
n_cols = 4  # Number of columns in the grid
n_rows = int(np.ceil(len(numeric_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows), sharex=True)
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].plot(algeria_df['Date'], algeria_df[col])
    axes[i].set_title(col)
    axes[i].set_ylabel(col)

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.xlabel('Date')
plt.tight_layout()
plt.show()

In [ ]:
# Plot UV Index for each year from 2001 to 2024 in subplots
years = range(2001, 2025)  # 24 years
fig, axes = plt.subplots(nrows=6, ncols=4, figsize=(20, 15))
axes = axes.flatten()

for i, year in enumerate(years):
    year_data = algeria_df[algeria_df['Date'].dt.year == year]
    axes[i].plot(year_data['Date'], year_data['ALLSKY_SFC_UV_INDEX'])
    axes[i].set_title(f'Year {year}')
    axes[i].set_xlabel('Date')
    axes[i].set_ylabel('UV Index')

plt.tight_layout()
plt.show()

### Overlapping

In [ ]:
import matplotlib.colors as mcolors

# Create a colormap from purple to red
cmap = mcolors.LinearSegmentedColormap.from_list("", ["purple", "red"])
norm = mcolors.Normalize(vmin=2001, vmax=2024)

plt.figure(figsize=(14, 8))

# Plot only for 2001 and 2024
for year in [2001, 2024]:
    year_data = algeria_df[algeria_df['Date'].dt.year == year]
    if not year_data.empty:
        day_of_year = year_data['Date'].dt.dayofyear
        uv_index = year_data['ALLSKY_SFC_UV_INDEX']
        color = cmap(norm(year))
        plt.plot(day_of_year, uv_index, color=color, alpha=0.7, linewidth=2, label=str(year))

plt.xlabel('Day of Year')
plt.ylabel('UV Index')
plt.title('UV Index Comparison: 2001 vs 2024')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Statistics

### Moving Average

In [ ]:
# Calculate yearly average UV index
algeria_df['Year'] = algeria_df['Date'].dt.year
yearly_avg_uv = algeria_df.groupby('Year')['ALLSKY_SFC_UV_INDEX'].mean()

# Exclude 2025
yearly_avg_uv = yearly_avg_uv[yearly_avg_uv.index != 2025]

# Create an appealing plot
plt.figure(figsize=(12, 6))
plt.plot(yearly_avg_uv.index, yearly_avg_uv.values, marker='o', linestyle='-', color='skyblue', linewidth=2, markersize=6)
plt.title('Evolution of Average UV Index by Year', fontsize=16, fontweight='bold')
plt.xlabel('Year', fontsize=14)
plt.ylabel('Average UV Index', fontsize=14)
plt.grid(True, alpha=0.3)
plt.xticks(yearly_avg_uv.index, rotation=45)
plt.tight_layout()
plt.show()

# Modelling

## Time Series Models

### Preparation

In [ ]:
from utils.ts import (
    MODELS, train_sarima, train_arima, train_ets, train_prophet, train_garch,
    train_naive, train_seasonal_naive, forecast_model, evaluate_forecast,
    get_default_params, PROPHET_AVAILABLE, GARCH_AVAILABLE
)

In [ ]:
ts_df = algeria_df.set_index("Date")[["ALLSKY_SFC_UV_INDEX"]].copy().sort_index()

train_size = int(len(ts_df) * 0.8)
train_ts = ts_df.iloc[:train_size]
test_ts  = ts_df.iloc[train_size:]

train_series = train_ts["ALLSKY_SFC_UV_INDEX"]
y_true = test_ts["ALLSKY_SFC_UV_INDEX"].values
h = len(test_ts)

print(f"Train size: {len(train_ts)}, Test size: {len(test_ts)}")

ts_results = {}

def _store_result(name, model, forecast):
    metrics = evaluate_forecast(y_true, forecast)
    ts_results[name] = {
        "model": model,
        "forecast": forecast,
        "RMSE": metrics["RMSE"],
        "MAE": metrics["MAE"],
    }
    print(f"{name} - RMSE: {metrics['RMSE']:.4f}, MAE: {metrics['MAE']:.4f}")

In [ ]:
train_series = train_series.asfreq("D")
test_series  = test_ts["ALLSKY_SFC_UV_INDEX"].asfreq("D")

# Fill any missing days (choose one policy)
train_series = train_series.interpolate(limit=7)
test_series  = test_series.interpolate(limit=7)

y_true = test_series.values
h = len(test_series)

### Models

In [ ]:
from utils.plot import plot_single_forecast, plot_residuals  

#### SARIMAX

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX

# -----------------------------
# 1) Prepare series (daily, regular)
# -----------------------------
y = algeria_df.set_index("Date")["ALLSKY_SFC_UV_INDEX"].sort_index()
y = y.asfreq("D")
y = y.interpolate(limit=7)  # handle small gaps

train_size = int(len(y) * 0.8)
train_y = y.iloc[:train_size]
test_y  = y.iloc[train_size:]
h = len(test_y)

# -----------------------------
# 2) Fourier terms for yearly seasonality
# -----------------------------
def fourier_terms(index, period=365.25, K=6):
    t = np.arange(len(index))
    X = {}
    for k in range(1, K + 1):
        X[f"sin{k}"] = np.sin(2 * np.pi * k * t / period)
        X[f"cos{k}"] = np.cos(2 * np.pi * k * t / period)
    return pd.DataFrame(X, index=index)

K = 8  # try 4/6/8 if needed
X_train = fourier_terms(train_y.index, K=K)
X_test  = fourier_terms(test_y.index,  K=K)

# -----------------------------
# 3) Fit SARIMAX (ARIMA + Fourier yearly seasonality)
# -----------------------------
model = SARIMAX(
    train_y,
    trend="t",                     
    exog=X_train,                  # yearly Fourier
    order=(1, 1, 1),
    seasonal_order=(1, 0, 1, 7),   # light weekly structure
    enforce_stationarity=False,
    enforce_invertibility=False,
)

res = model.fit(disp=False, maxiter=100)

# -----------------------------
# 4) Forecast on test horizon
# -----------------------------
forecast = res.forecast(steps=h, exog=X_test)
forecast = np.asarray(forecast)

# -----------------------------
# 5) Evaluate + plot using your utilities
# -----------------------------
from utils.ts import evaluate_forecast
metrics = evaluate_forecast(test_y.values, forecast)
print(f"SARIMAX_FOURIER_K{K} - RMSE: {metrics['RMSE']:.4f}, MAE: {metrics['MAE']:.4f}")


In [ ]:
# Your plot utility
plot_single_forecast(
    train=train_y,
    test=test_y,
    forecast=forecast,
    variable_name="ALLSKY_SFC_UV_INDEX",
    lookback_days=365,
    model_name=f"SARIMAX_FOURIER_K{K}"
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

# --- 1) Build (doy, uv, year) ---
df = algeria_df.copy()
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date")

# drop Feb 29 to force 365-day years
df = df[~((df["Date"].dt.month == 2) & (df["Date"].dt.day == 29))]

df["year"] = df["Date"].dt.year
df["doy"]  = df["Date"].dt.dayofyear

# pivot: each year is a curve over day-of-year
Y = (df.pivot_table(index="year", columns="doy", values="ALLSKY_SFC_UV_INDEX", aggfunc="mean")
    .reindex(columns=range(1, 366)))

# --- 2) Density background: where values occur most often (2D histogram) ---
# Flatten all points (doy, uv) across years
doy_vals = np.tile(np.arange(1, 366), len(Y.index))
uv_vals = Y.values.reshape(-1)

mask = ~np.isnan(uv_vals)
doy_vals = doy_vals[mask]
uv_vals  = uv_vals[mask]

# bins (adjust uv_bins for smoother/rougher density)
uv_min, uv_max = np.nanmin(uv_vals), np.nanmax(uv_vals)
doy_bins = np.arange(1, 367)  # 365 bins
uv_bins = np.linspace(uv_min, uv_max, 120)

H, xedges, yedges = np.histogram2d(doy_vals, uv_vals, bins=[doy_bins, uv_bins])
H = H.T  # so y is vertical

# log-scale density (optional but usually better)
H_plot = np.log1p(H)

# --- 3) Forecasted expected year line (must be 365 daily points) ---
# If your forecast is longer, we take first 365; if shorter, adjust accordingly.
forecast_year = np.asarray(forecast[:365])
doy = np.arange(1, 366)

# --- 4) Plot ---
fig, ax = plt.subplots(figsize=(14, 6))

# Density background with appealing colormap
im = ax.pcolormesh(xedges, yedges, H_plot, shading="auto", alpha=0.6, cmap='Blues')

# Add colorbar for density
cbar = plt.colorbar(im, ax=ax, orientation='vertical', shrink=0.8)
cbar.set_label('Log Density')

# Spaghetti: all historical years with light color
for yr in Y.index:
    ax.plot(np.arange(1, 366), Y.loc[yr].values, linewidth=1, alpha=0.1, color='lightgray')

# Forecast expected year with bold color
ax.plot(doy, forecast_year, linewidth=4, color='darkblue', label="Forecast (expected year)")

ax.set_title("UV by Day-of-Year: Density + All Years + Forecast")
ax.set_xlabel("Day of Year")
ax.set_ylabel("UV Index")
ax.grid(True, alpha=0.2)
ax.legend()
plt.tight_layout()
plt.show()


#### Summary

# XGBoost Model

In [ ]:
import time
target_col = "ALLSKY_SFC_UV_INDEX"

# Create clean dataset
ml_df = algeria_df.dropna().copy()
ml_df["Date"] = pd.to_datetime(ml_df["Date"])
ml_df = ml_df.set_index("Date").sort_index()

# Exclude target and related UV columns (avoid data leakage)
exclude_cols = [target_col, "ALLSKY_SFC_UVA", "ALLSKY_SFC_UVB"]
base_feature_cols = [col for col in ml_df.columns if col not in exclude_cols]

# ===== FEATURE ENGINEERING =====
# 1. Temporal Features
ml_df['day_of_year'] = ml_df.index.dayofyear
ml_df['month'] = ml_df.index.month
ml_df['week'] = ml_df.index.isocalendar().week.astype(int)
# Cyclic encoding for seasonal patterns
ml_df['day_sin'] = np.sin(2 * np.pi * ml_df['day_of_year'] / 365)
ml_df['day_cos'] = np.cos(2 * np.pi * ml_df['day_of_year'] / 365)
ml_df['month_sin'] = np.sin(2 * np.pi * ml_df['month'] / 12)
ml_df['month_cos'] = np.cos(2 * np.pi * ml_df['month'] / 12)
# 2. Lag Features (UV tends to have temporal autocorrelation)
for lag in [1, 2, 3, 7, 14]:
    ml_df[f'T2M_lag{lag}'] = ml_df['T2M'].shift(lag)
    ml_df[f'RH2M_lag{lag}'] = ml_df['RH2M'].shift(lag)
    ml_df[f'TO3_lag{lag}'] = ml_df['TO3'].shift(lag)
    
# 3. Rolling Statistics (capture trends)
for window in [7, 14, 30]:
    ml_df[f'T2M_roll_mean{window}'] = ml_df['T2M'].rolling(window=window).mean()
    ml_df[f'T2M_roll_std{window}'] = ml_df['T2M'].rolling(window=window).std()
    ml_df[f'RH2M_roll_mean{window}'] = ml_df['RH2M'].rolling(window=window).mean()
    if 'ALLSKY_SFC_SW_DWN' in ml_df.columns:
        ml_df[f'ALLSKY_SFC_SW_DWN_roll_mean{window}'] = ml_df['ALLSKY_SFC_SW_DWN'].rolling(window=window).mean()
# 4. Interaction Features (only if columns exist)
ml_df['temp_humidity_interaction'] = ml_df['T2M'] * ml_df['RH2M']
# Check if columns exist before creating interaction
if 'ALLSKY_SFC_SW_DWN' in ml_df.columns and 'ALLSKY_KT' in ml_df.columns:
    ml_df['solar_clearness'] = ml_df['ALLSKY_SFC_SW_DWN'] * ml_df['ALLSKY_KT']
elif 'ALLSKY_SFC_SW_DWN' in ml_df.columns and 'CLRSKY_SFC_SW_DWN' in ml_df.columns:
    # Alternative: ratio of actual to clear-sky radiation (similar concept)
    ml_df['solar_clearness'] = ml_df['ALLSKY_SFC_SW_DWN'] / (ml_df['CLRSKY_SFC_SW_DWN'] + 0.001)
# Drop rows with NaN created by lag/rolling features
ml_df = ml_df.dropna()
# Updated feature columns
exclude_cols_updated = [target_col, "ALLSKY_SFC_UVA", "ALLSKY_SFC_UVB"]
feature_cols = [col for col in ml_df.columns if col not in exclude_cols_updated]
print(f"Dataset size after feature engineering: {len(ml_df)} rows")
print(f"Total features: {len(feature_cols)}")
print(f"\nAvailable columns: {list(ml_df.columns)}")

# Same 80-20 time-based split
train_size = int(len(ml_df) * 0.8)
train_ml = ml_df.iloc[:train_size]
test_ml = ml_df.iloc[train_size:]

X_train = train_ml[feature_cols]
X_test = test_ml[feature_cols]
y_train = train_ml[target_col]
y_test = test_ml[target_col]

print(f"Train: {len(X_train)}, Test: {len(X_test)}")
print(f"Date range - Train: {train_ml.index.min()} to {train_ml.index.max()}")
print(f"Date range - Test: {test_ml.index.min()} to {test_ml.index.max()}")

# %% [markdown]
# ### Hyperparameter Tuning with GridSearchCV

# %%
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, RandomizedSearchCV

# TimeSeriesSplit for proper cross-validation
tscv = TimeSeriesSplit(n_splits=5)

# Parameter grid for XGBoost
param_grid = {
    'n_estimators': [200, 300, 500],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 5, 10]
}

# For faster tuning, use RandomizedSearchCV
print("Performing hyperparameter tuning...")
start_time = time.time()

xgb_base = xgb.XGBRegressor(random_state=42, n_jobs=-1)

# Use RandomizedSearchCV for faster search
search = RandomizedSearchCV(
    xgb_base,
    param_distributions=param_grid,
    n_iter=50,  # Number of parameter combinations to try
    cv=tscv,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search.fit(X_train, y_train)

tuning_time = time.time() - start_time
print(f"\nTuning completed in {tuning_time:.2f} seconds")
print(f"\nBest parameters: {search.best_params_}")
print(f"Best CV RMSE: {-search.best_score_:.4f}")

# %% [markdown]
# ### Train Final Model with Best Parameters

# %%
# Use best parameters found
best_params = search.best_params_

# Train final model with early stopping
xgb_best = xgb.XGBRegressor(
    **best_params,
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=50
)

print("Training final model with early stopping...")
start_time = time.time()

xgb_best.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=False
)

final_training_time = time.time() - start_time
print(f"Training completed in {final_training_time:.2f} seconds")

# Generate predictions
xgb_forecast = xgb_best.predict(X_test)

# Calculate metrics
from utils.ts import evaluate_forecast
xgb_metrics = evaluate_forecast(y_test.values, xgb_forecast)

print(f"\nOptimized XGBoost - RMSE: {xgb_metrics['RMSE']:.4f}, MAE: {xgb_metrics['MAE']:.4f}")

# Store results
xgb_results = {}
xgb_results["XGBoost_Optimized"] = {
    "model": xgb_best,
    "forecast": xgb_forecast,
    "RMSE": xgb_metrics["RMSE"],
    "MAE": xgb_metrics["MAE"],
}

# %%
from sklearn.metrics import r2_score

# Training set predictions
y_pred_train = xgb_best.predict(X_train)
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_train))
train_mae = mean_absolute_error(y_train, y_pred_train)
train_r2 = r2_score(y_train, y_pred_train)

# Test set metrics
test_rmse = xgb_metrics["RMSE"]
test_mae = xgb_metrics["MAE"]
test_r2 = r2_score(y_test, xgb_forecast)

# MAPE and Accuracy
mape = np.mean(np.abs((y_test.values - xgb_forecast) / y_test.values)) * 100
accuracy = 100 - mape

print("=" * 60)
print("           XGBoost OPTIMIZED Model Results")
print("=" * 60)
print(f"\n{'Metric':<15} {'Training':<15} {'Test':<15}")
print("-" * 45)
print(f"{'RMSE':<15} {train_rmse:<15.4f} {test_rmse:<15.4f}")
print(f"{'MAE':<15} {train_mae:<15.4f} {test_mae:<15.4f}")
print(f"{'R²':<15} {train_r2:<15.4f} {test_r2:<15.4f}")
print("-" * 45)
print(f"\nTest MAPE: {mape:.2f}%")
print(f"Test Accuracy: {accuracy:.2f}%")
print("=" * 60)

# %% [markdown]
# ### Feature Importance Analysis

# %%
# Get feature importance
feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': xgb_best.feature_importances_
}).sort_values('Importance', ascending=False)

print("Top 15 Most Important Features:")
print(feature_importance.head(15).to_string(index=False))

# Plot feature importance  
fig, ax = plt.subplots(figsize=(12, 8))
top_n = 20
top_features = feature_importance.head(top_n)

colors = plt.cm.viridis(np.linspace(0.3, 0.9, top_n))
bars = ax.barh(top_features['Feature'][::-1], 
               top_features['Importance'][::-1],
               color=colors[::-1])

ax.set_xlabel('Importance', fontsize=12)
ax.set_ylabel('Feature', fontsize=12)
ax.set_title(f'XGBoost Feature Importance (Top {top_n})', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()